In [ ]:
"""
Diferencias principales con el código anterior: 

- Cambio al reemplazar os.system('cls/'clear') por clear_output(wait=True) de IPython.display  
- Código ANSI para borrar lineas se elimina, ya que ahora los errores de validación se imprimen abajo.

* Código ANSI no es válido en jupyter, es por que se reemplaza por clear_output que limpia toda la celda de una vez.  

"""
import json, random, time
from IPython.display import clear_output # le dice directamente a jupyter que limpie la celda, sin necesidad de depender del sistema operativo

LETRAS_VALIDAS = "qwertyuiopasdfghjklñzxcvbnmáéíóú"
MAX_ERRORES = 10

def imprimir(etapas, errores, tabla):
    clear_output(wait=True) # Imprime el nuevo estado del juego y elimina el anterior, funciona como loop.
    print(etapas[errores])
    linea = ' '.join(tabla) # Une los elementos de la tabla con separación de un espacio 
    print(f"\n  {linea}\n") 
    print(f"  Intentos fallidos  : {errores}/{MAX_ERRORES}")
    print(f"  Intentos restantes : {MAX_ERRORES - errores}")

    time.sleep(0.05) #Descubrimos que a veces falla el clear_output si va seguido de un input, por tanto, le daremos un tiempo para evitar el problema
    return linea


def normalizar(c): 
    return 'a' if c=='á' else 'e' if c=='é' else 'i' if c=='í' else 'o' if c=='ó' else 'u' if c=='ú' else c # Valida letras normales y sus tildes.

def valida(usadas): # Función que valida que la letra del jugador sea correcta antes de ser utilizada en el juego. 
    while True:
        c = input("Adivinanza: ").lower().strip() 

        if len(c) != 1: # ¿Qué ocurre si el jugador ingresó mas de un carácter? 
            imprimir(etapas,errores,tabla)
            print("Error: ingresa solo UNA letra a la vez.") # Mostrar error
            continue
        if c not in LETRAS_VALIDAS: # ¿Qué ocurre si la letra no está en el abecedario?
            imprimir(etapas,errores,tabla)
            print(f"Error: '{c}' no es una letra válida del abecedario.") # Imprimir error
            continue
        cn = normalizar(c) # Primero normaliza la letra (saca la tilde). 
        if cn in usadas: 
            imprimir(etapas,errores,tabla)
            print(f"Error: la letra '{c}' ya fue ingresada antes.") # luego revisa si fue usada antes, en caso de ser así muestra error.
            continue

        return cn

with open("spanish.json", "r", encoding="utf-8") as f:  
    data = json.load(f) # Abre y carga los datos json (cierra el archivo automaticamente) 
with open("hangman.json", "r", encoding="utf-8") as f:
    ascii_art = json.load(f) # Abre y carga los datos, pero el resultado queda en el diccionario. 

# Verificación para que hangman.json tenga las suficientes etapas antes de empezar.
etapas = ascii_art["Hangman"] # Atajo para no escribir ascii_art["Hangman"] cada vez.
if len(etapas) < MAX_ERRORES + 1: # Verifica que la lista tenga 11 elementos. 
    raise ValueError( # Si esto se cumple mostrar error. 
        f"hangman.json tiene solo {len(etapas)} etapas en 'Hangman', "
        f"pero se necesitan {MAX_ERRORES + 1} (índices 0 a {MAX_ERRORES})."
    )

while True: 
    real = random.choice(data).lower() # Elige aleatoriamente una palabra hasta que esta sea de 4 o más letras. 
    if len(real) >= 4:
        break

palabra_norm = ''.join(normalizar(x) for x in real) # Lee las palabras sin tilde, las normaliza y las une en un string. 
tabla = ['_'] * len(real) # Se reemplaza el guión por la letra una vez que la letra esté correcta.

usadas      = set() # En un conjunto vacío se guardan todas las letras ya ingresadas.
incorrectas = [] # Guarda todos los errores.  
errores     = 0 

while True:
    linea = imprimir(etapas,errores,tabla)

    if '_' not in linea: # Si todas las lineas estan utilizadas por letras reales...
        print("\n  ¡Ganaste! ¡Adivinaste la palabra!\n")
        break

    if errores >= MAX_ERRORES: # LLegas a los 10 intentos fallidos.
        print(f"\n  ¡Perdiste! La palabra correcta era: {real.capitalize()}\n")
        break

    if incorrectas:
        print(f"  Letras incorrectas : {', '.join(incorrectas)}")

    print()
    caracter = valida(usadas)
    usadas.add(caracter)

    if caracter in palabra_norm: # Revela la posicion y la letra a la vez. 
        for i, x in enumerate(real): 
            if caracter == normalizar(x):
                tabla[i] = x
    else:
        errores += 1
        incorrectas.append(caracter)

  +---+
  |   |
  O   |
 /|\  |
 /    |
      |

  _ a _ i ó _

  Intentos fallidos  : 8/10
  Intentos restantes : 2
Error: '4' no es una letra válida del abecedario.
